In [1]:
import os 
from dotenv import load_dotenv

if os.environ.get("API_KEY") is None:
    raise ValueError("API_KEY environment variable is not set. Please set it in your .env file.")

else:
    print("API_KEY environment variable is set.")

API_KEY environment variable is set.


Test API

In [2]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model= "gpt-5-nano", temperature=0, api_key = os.environ.get("API_KEY"))

response = llm.invoke("Hello what is the capital of France?")

print(response.content)

The capital of France is Paris.


### RAG IMPLEMENTATION WITH  ADDED PDFS 

### Step 1: Extracting Text From PDFS

In [3]:
from langchain_community.document_loaders import PyPDFLoader

pdf_admin_path = "/Users/jessepepple/Downloads/RAG_Project_Main/Docs/fabric-admin.pdf"

Loader = PyPDFLoader(pdf_admin_path)

docs = Loader.load()

/var/folders/90/v950rdks7f93s0k0r2wr6kfh0000gn/T/ipykernel_48769/1256869173.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Creating our Metadata

In [5]:
for i in docs:
    i.metadata = {
        "source" : "fabric-admin.pdf",
        "developer" : "Microsoft"
    }

In [6]:
docs[0].metadata

{'source': 'fabric-admin.pdf', 'developer': 'Microsoft'}

### Step 2 Chunking

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

chunks = splitter.split_documents(docs)

chunks

[Document(metadata={'source': 'fabric-admin.pdf', 'developer': 'Microsoft'}, page_content='Tell us about your PDF experience.\nMicrosoft Fabric documentation for\nadmins\nLearn about the Microsoft Fabric admin settings, options, and tools.\nFabric in your organization\nｅ OVERVIEW\nWhat is Microsoft Fabric admin?\nWhat is the admin portal?\nｂ GET STARTED\nEnable Fabric for your organization\nRegion availability\nFind your Fabric home region\nｃ HOW-TO GUIDE\nUnderstand Fabric admin roles\nｉ REFERENCE\nGovernance documentation\nSecurity documentation\nTools and settings\nｅ OVERVIEW\nAbout tenant settings\nｃ HOW-TO GUIDE\nSet up git integration\nSet up item certification'),
 Document(metadata={'source': 'fabric-admin.pdf', 'developer': 'Microsoft'}, page_content='Configure notifications\nSet up metadata scanning\nEnable content certification\nEnable service principal authentication\nConfigure Multi-Geo support\nMonitoring and management\nｅ OVERVIEW\nWhat is the admin monitoring workspace?\

### Step 3 Create Embeddings 

In [8]:
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(model= "text-embedding-3-small", api_key = os.environ.get("API_KEY"))
embedding_model

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x11727aa50>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x11727acf0>, model='text-embedding-3-small', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

### Step 4 Create and Store Embeddings In The Existing Local Vector Store

In [9]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma(persist_directory="/Users/jessepepple/Downloads/RAG_Project_Main/Vector", embedding_function=embedding_model)

/var/folders/90/v950rdks7f93s0k0r2wr6kfh0000gn/T/ipykernel_48769/441968580.py:3: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(persist_directory="/Users/jessepepple/Downloads/RAG_Project_Main/Vector", embedding_function=embedding_model)


In [10]:
vectorstore.add_documents(chunks)

['3c5d7e43-6841-42fd-92fe-dbf9456fd1c5',
 'c5ce7e2d-4f79-4e16-929c-84aec4542a03',
 '391f76e9-8ff7-4133-8001-f50c6594855f',
 'aabf85d6-4acc-4247-b7e9-f0c25050ac8a',
 '10a7d19e-e836-44ac-95d8-eec8ad8688d6',
 '9da9ee31-e558-48df-91fa-daf89f2b8099',
 '7b98fc3b-3a5a-4f98-a6cd-cc0de9691098',
 '395a88d7-b7ea-4b90-a7af-922483616f96',
 '67035f9b-121e-46d5-9226-677727af195e',
 'd992c082-2d91-4e1c-b432-ce51302e23f4',
 'bf02695d-9d0d-461f-bbab-5df5ec712190',
 'fa268237-f58e-4619-b862-235c1b2d239c',
 '6958b485-f52e-49c9-90c7-b10b9730f2d6',
 'ccfdd7d1-5756-4f5a-96e7-4ef6a0b7ce2f',
 '0262ee3d-7792-46c5-9758-65a43c0c5f8d',
 'a4bdbfa8-bfbc-449d-8361-0c58602da611',
 '597cfcfd-0bd4-43ae-82fa-9ab10d95440e',
 '35875522-55bf-4eba-8cdc-78838a39e221',
 'df4ad37a-a89c-43b1-a824-d276f1183b19',
 'ff660662-3200-43bc-b185-69652aa8fda7',
 '555fb106-c06b-4a84-8b6a-39d7b62369d5',
 'cd5bdd10-b515-4f2f-8614-979602b96823',
 '6929cac9-a661-4a98-ab6c-eba363fc3846',
 'dc243c61-24b5-4cd2-ac74-9221aedcabd7',
 '3fe1c465-96f4-

### Step 5: Semantic Search 

In [33]:
vectorstore.similarity_search("What is OneLake?", k=3)

[Document(metadata={'creator': 'Microsoft Learn', 'creationdate': '2025-12-12T17:01:10+00:00', 'title': 'fabric onelake | Microsoft Learn', 'producer': 'Microsoft Learn PDF 1.0.25309.01', 'moddate': '2025-12-12T17:01:10+00:00', 'page_label': '3', 'page': 2, 'source': '/Users/jessepepple/Downloads/RAG_Project_Main/Docs/fabric-onelake.pdf', 'total_pages': 382}, page_content='OneLake, the OneDrive for data\nArticle • 07/25/2024\nOneLake is a single, unified, logical data lake for your whole organization. A data Lake\nprocesses large volumes of data from various sources. Like OneDrive, OneLake comes\nautomatically with every Microsoft Fabric tenant and is designed to be the single place\nfor all your analytics data. OneLake brings customers:\nOne data lake for the entire organization\nOne copy of data for use with multiple analytical engines\nBefore OneLake, it was easier for customers to create multiple lakes for different\nbusiness groups rather than collaborating on a single lake, even 

Talk To LLM

In [11]:
context = vectorstore.similarity_search("What is Microsoft Fabric Admin?", k=3)
context

[Document(metadata={'source': 'fabric-admin.pdf', 'developer': 'Microsoft'}, page_content='What is Microsoft Fabric admin?\n07/01/2024\nMicrosoft Fabric admin is the management of the organization-wide settings that control how\nMicrosoft Fabric works. Users that are assigned to admin roles configure, monitor, and\nprovision organizational resources. This article provides an overview of admin roles, tasks, and\ntools to help you get started.\nThere are several roles that work together to administer Microsoft Fabric for your organization.\nMost admin roles are assigned in the Microsoft 365 admin portal or by using PowerShell. The\ncapacity admin roles are assigned when the capacity is created. To learn more about each of\nthe admin roles, see About admin roles. To learn how to assign admin roles, see Assign admin\nroles.\nThis section lists the Microsoft 365 admin roles and the tasks they can perform.\nGlobal administrator\nUnlimited access to all management features for the organizatio

In [12]:
response  = llm.invoke(f"Answer the question: regarding Microsoft Fabric Admin: {context}")
response.content

'Here’s a concise answer based on the Microsoft Fabric Admin documentation you shared.\n\nWhat is Microsoft Fabric admin?\n- It is the set of organization-wide settings and controls that govern how Microsoft Fabric operates. Admins configure, monitor, and provision resources, manage security and governance, control feature access, and delegate admin responsibilities as needed.\n\nKey concepts\n- Admin roles: Different roles exist to share governance duties. Roles can be assigned in the Microsoft 365 admin portal or via PowerShell. Some roles are assigned when a fabric capacity is created.\n- Delegation: Admin responsibilities can be split into groups to avoid bottlenecks and to align with organizational policy and compliance needs.\n- Access: Fabric administration is distinct from full Microsoft 365 admin rights; dedicated Fabric or Power Platform admin roles grant the necessary access to Fabric settings without giving all Microsoft 365 permissions.\n\nCommon admin roles and their typi